# AE101 MART 노트북 : 차원·Fact 생성 + ETL (CORE → MART)

이 노트북은 **MART 만** 담당합니다. 정제(CORE)까지 끝난 뒤 실행하세요.

**사전 조건** : 메인 워크스루 노트북에서 STEP 1(DDL)~STEP 6(감사 기록)까지 실행되어 `CORE.ORDERS`, `CORE.SALES_LINE_PREP`, `CORE.PAYMENT_PREP`, `CORE.ORDER_QUALITY`, `CORE.RUN_CTX`, `CORE.RUN_SUMMARY` 가 있어야 합니다.

| 순서 | 내용 |
|---|---|
| A | 사전 점검 |
| B | MART 차원·Fact 테이블 생성 (여러 번 실행해도 안전) |
| C | ETL : SCD2 차원 → Fact MERGE → 품질 검사 → 상태 확정 |
| D | 결과 확인 |

> **노트북 방식의 차이**: 노트북은 셀마다 따로 실행되므로 MART 반영을 하나의 트랜잭션 블록 대신 아래 셀로 나눴습니다. 각 셀의 MERGE/UPDATE 는 같은 `load_id` 로 다시 실행해도 결과가 같습니다(멱등). 대신 **검사 실패 시 자동 ROLLBACK 은 없습니다.** `FAILED` 가 나오면 원인을 고친 뒤 7-1a 부터 다시 실행하세요. 되돌려야 하면 `CREATE OR REPLACE TABLE … CLONE … AT(TIMESTAMP => <run_ts>)` 같은 Time Travel 을 쓰세요.

## A. 사전 점검

In [ ]:
USE ROLE ACCOUNTADMIN;
USE DATABASE AE101_SALES_ENT;
SELECT 'CORE.ORDERS' AS tbl, COUNT(*) AS n FROM CORE.ORDERS
UNION ALL SELECT 'CORE.SALES_LINE_PREP', COUNT(*) FROM CORE.SALES_LINE_PREP
UNION ALL SELECT 'CORE.PAYMENT_PREP', COUNT(*) FROM CORE.PAYMENT_PREP
UNION ALL SELECT 'CORE.ORDER_QUALITY', COUNT(*) FROM CORE.ORDER_QUALITY
UNION ALL SELECT 'CORE.RUN_CTX', COUNT(*) FROM CORE.RUN_CTX;   -- 테이블이 없다는 오류가 나면 메인 노트북 STEP 2~6 을 먼저 실행하세요.

## B. MART 테이블 생성

### B-1. 차원 (대리키 + SCD2 + Unknown 멤버)

In [ ]:
-- 1-3. MART 차원 (3-2, 3-3) : 대리키 + SCD Type 2
CREATE SEQUENCE IF NOT EXISTS MART.SEQ_CUSTOMER_KEY START = 1 INCREMENT = 1;
CREATE SEQUENCE IF NOT EXISTS MART.SEQ_PRODUCT_KEY  START = 1 INCREMENT = 1;

CREATE TABLE IF NOT EXISTS MART.DIM_CUSTOMER (
  customer_key   NUMBER DEFAULT MART.SEQ_CUSTOMER_KEY.NEXTVAL NOT NULL,  -- 대리키
  customer_id    VARCHAR NOT NULL,                                       -- 업무 키
  customer_name  VARCHAR,
  email          VARCHAR,                                                -- PII: STEP 9 에서 마스킹
  region         VARCHAR,                                                -- STEP 9 행 수준 보안 기준
  valid_from     TIMESTAMP_NTZ NOT NULL,
  valid_to       TIMESTAMP_NTZ,                                          -- NULL = 현재 유효
  is_current     BOOLEAN NOT NULL,
  hash_diff      VARCHAR,                                                -- 변경 감지용 해시
  load_id        VARCHAR,
  CONSTRAINT PK_DIM_CUSTOMER PRIMARY KEY (customer_key));

CREATE TABLE IF NOT EXISTS MART.DIM_PRODUCT (
  product_key    NUMBER DEFAULT MART.SEQ_PRODUCT_KEY.NEXTVAL NOT NULL,
  product_id     VARCHAR NOT NULL,
  product_name   VARCHAR,
  category       VARCHAR,
  valid_from     TIMESTAMP_NTZ NOT NULL,
  valid_to       TIMESTAMP_NTZ,
  is_current     BOOLEAN NOT NULL,
  hash_diff      VARCHAR,
  load_id        VARCHAR,
  CONSTRAINT PK_DIM_PRODUCT PRIMARY KEY (product_key));

-- Unknown 멤버(-1): 참조 대상이 없는 Fact 도 버리지 않고 적재하기 위한 자리
INSERT INTO MART.DIM_CUSTOMER (customer_key, customer_id, customer_name, email, region, valid_from, valid_to, is_current, hash_diff, load_id)
SELECT -1, '-1', 'Unknown', NULL, 'Unknown', '1900-01-01'::TIMESTAMP_NTZ, NULL, TRUE, NULL, 'SEED'
WHERE NOT EXISTS (SELECT 1 FROM MART.DIM_CUSTOMER WHERE customer_key = -1);

INSERT INTO MART.DIM_PRODUCT (product_key, product_id, product_name, category, valid_from, valid_to, is_current, hash_diff, load_id)
SELECT -1, '-1', 'Unknown', 'Unknown', '1900-01-01'::TIMESTAMP_NTZ, NULL, TRUE, NULL, 'SEED'
WHERE NOT EXISTS (SELECT 1 FROM MART.DIM_PRODUCT WHERE product_key = -1);

### B-2. 날짜·상태 차원

In [ ]:
-- 전체 달력 2020-01-01 ~ 2030-12-31 (4,018일) + Unknown(-1)
CREATE OR REPLACE TABLE MART.DIM_DATE AS
WITH days AS (
  SELECT DATEADD(day, ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, '2020-01-01'::DATE) AS d
  FROM TABLE(GENERATOR(ROWCOUNT => 4018)))
SELECT YEAR(d) * 10000 + MONTH(d) * 100 + DAY(d)       AS date_key,
       d                                               AS calendar_date,
       YEAR(d)                                         AS year,
       QUARTER(d)                                      AS quarter,
       MONTH(d)                                        AS month,
       TO_CHAR(d, 'YYYY-MM')                           AS year_month,
       DAY(d)                                          AS day_of_month,
       DAYOFWEEKISO(d)                                 AS day_of_week,     -- 1=월 ... 7=일
       DAYNAME(d)                                      AS day_name,
       DAYOFWEEKISO(d) IN (6, 7)                       AS is_weekend
FROM days
UNION ALL
SELECT -1, '1900-01-01'::DATE, 1900, 1, 1, 'Unknown', 1, 1, 'Unk', FALSE;

CREATE OR REPLACE TABLE MART.DIM_ORDER_STATUS AS
SELECT * FROM (VALUES
  ( 1, 'completed', '완료', TRUE),
  ( 2, 'pending',   '대기', FALSE),
  ( 3, 'cancelled', '취소', TRUE),
  (-1, 'unknown',   '알 수 없음', FALSE)
) AS v(order_status_key, status_code, status_name, is_final);

CREATE OR REPLACE TABLE MART.DIM_PAYMENT_STATUS AS
SELECT * FROM (VALUES
  ( 1, 'succeeded', '성공', TRUE),
  ( 2, 'failed',    '실패', FALSE),
  ( 3, 'pending',   '대기', FALSE),
  (-1, 'unknown',   '알 수 없음', FALSE)
) AS v(payment_status_key, status_code, status_name, is_success);
-- 참고: 원본 payments.csv 에는 결제 수단 컬럼이 없어 dim_payment_method 대신 결제 상태 차원을 둡니다.

### B-3. Fact 3종

In [ ]:
-- 1-4. MART Fact (3-2)
-- 입도: 주문 상세 1행. 매출 분석의 중심 Fact
CREATE TABLE IF NOT EXISTS MART.FACT_SALES_LINE (
  order_item_id       VARCHAR NOT NULL,      -- degenerate key (입도 식별자)
  order_id            VARCHAR NOT NULL,      -- degenerate dimension
  customer_key        NUMBER  NOT NULL,      -- 주문일 시점의 고객 버전
  product_key         NUMBER  NOT NULL,      -- 주문일 시점의 상품 버전
  order_date_key      NUMBER  NOT NULL,
  order_status_key    NUMBER  NOT NULL,
  order_date          DATE,
  quantity            NUMBER(38,0),
  unit_price          NUMBER(38,0),
  line_amount         NUMBER(38,0),
  is_revenue_eligible BOOLEAN NOT NULL,      -- 매출 인식 여부 (STEP 5 규칙)
  exclusion_reason    VARCHAR,               -- 매출 제외 사유
  is_deleted          BOOLEAN NOT NULL,      -- 소프트 삭제
  load_id             VARCHAR,
  updated_at          TIMESTAMP_NTZ,
  CONSTRAINT PK_FACT_SALES_LINE PRIMARY KEY (order_item_id),
  CONSTRAINT FK_SL_CUSTOMER FOREIGN KEY (customer_key) REFERENCES MART.DIM_CUSTOMER (customer_key),
  CONSTRAINT FK_SL_PRODUCT  FOREIGN KEY (product_key)  REFERENCES MART.DIM_PRODUCT (product_key));

-- 입도: 결제 시도 1건
CREATE TABLE IF NOT EXISTS MART.FACT_PAYMENT (
  payment_id          VARCHAR NOT NULL,
  order_id            VARCHAR,
  customer_key        NUMBER  NOT NULL,
  paid_date_key       NUMBER  NOT NULL,
  payment_status_key  NUMBER  NOT NULL,
  paid_at             TIMESTAMP_NTZ,
  payment_amount      NUMBER(38,0),
  has_valid_order     BOOLEAN NOT NULL,
  is_deleted          BOOLEAN NOT NULL,
  load_id             VARCHAR,
  updated_at          TIMESTAMP_NTZ,
  CONSTRAINT PK_FACT_PAYMENT PRIMARY KEY (payment_id));

-- 입도: 주문 1건 (Accumulating Snapshot). 주문의 진행 단계를 한 행에 누적
CREATE TABLE IF NOT EXISTS MART.FACT_ORDER_LIFECYCLE (
  order_id                     VARCHAR NOT NULL,
  customer_key                 NUMBER  NOT NULL,
  order_date_key               NUMBER  NOT NULL,
  first_success_paid_date_key  NUMBER  NOT NULL,   -- 성공 결제 전이면 -1
  order_status_key             NUMBER  NOT NULL,
  order_date                   DATE,
  first_success_paid_at        TIMESTAMP_NTZ,
  order_amount                 NUMBER(38,0),       -- 헤더 금액은 주문 입도 Fact 에만 둡니다 (이중 집계 방지)
  line_count                   NUMBER,
  line_amount_total            NUMBER(38,0),
  success_payment_count        NUMBER,
  success_payment_amount       NUMBER(38,0),
  days_to_first_success_payment NUMBER,
  is_revenue_eligible          BOOLEAN NOT NULL,
  exclusion_reason             VARCHAR,
  is_deleted                   BOOLEAN NOT NULL,
  load_id                      VARCHAR,
  updated_at                   TIMESTAMP_NTZ,
  CONSTRAINT PK_FACT_ORDER_LIFECYCLE PRIMARY KEY (order_id));
-- 참고: Snowflake 의 PK/FK 는 문서·BI 도구용 "선언"이며 강제되지 않습니다. 그래서 STEP 7 에서 직접 검사합니다.

## C. ETL (CORE → MART)

### 7-0. 실행 컨텍스트 복원 + 가드
`CORE.RUN_CTX` 의 `load_id`/`run_ts` 를 세션 변수로 불러오고, 반영할 정상 주문이 있는지 확인합니다.

In [ ]:
-- 새 세션(노트북 재시작 등)이면 변수가 사라집니다. CORE.RUN_CTX 에서 다시 읽어 옵니다.
SET load_id = (SELECT load_id FROM CORE.RUN_CTX);
SET run_ts  = (SELECT run_ts  FROM CORE.RUN_CTX);
SELECT $load_id AS load_id, $run_ts AS run_ts;

In [ ]:
-- 이번 배치에 정상 주문이 있어야 MART 를 갱신합니다. 0 이면 아래 7-1 이후 셀을 실행하지 마세요.
SELECT $load_id AS load_id, $run_ts AS run_ts,
       (SELECT COUNT(*) FROM CORE.ORDERS) AS accepted_orders,
       IFF((SELECT COUNT(*) FROM CORE.ORDERS) = 0, '중단: 정상 주문 0건', 'OK') AS guard;

### 7-1a. SCD2 : DIM_CUSTOMER
바뀐 고객은 기존 버전을 닫고(`valid_to`) 새 버전을 추가합니다. 같은 배치로 다시 실행해도 결과가 같습니다(멱등).

In [ ]:
-- 7-1. SCD2 : dim_customer
UPDATE MART.DIM_CUSTOMER
   SET valid_to = $run_ts, is_current = FALSE
  FROM CORE.CUSTOMERS s
 WHERE MART.DIM_CUSTOMER.customer_id = s.customer_id
   AND MART.DIM_CUSTOMER.is_current
   AND MART.DIM_CUSTOMER.customer_key <> -1
   AND MART.DIM_CUSTOMER.hash_diff <> s.hash_diff;

INSERT INTO MART.DIM_CUSTOMER (customer_id, customer_name, email, region, valid_from, valid_to, is_current, hash_diff, load_id)
SELECT s.customer_id, s.customer_name, s.email, s.region,
       CASE WHEN h.customer_id IS NULL THEN '1900-01-01'::TIMESTAMP_NTZ ELSE $run_ts END,  -- 처음 보는 고객은 과거 전체에 유효
       NULL, TRUE, s.hash_diff, $load_id
FROM CORE.CUSTOMERS s
LEFT JOIN (SELECT DISTINCT customer_id FROM MART.DIM_CUSTOMER) h ON h.customer_id = s.customer_id
WHERE NOT EXISTS (SELECT 1 FROM MART.DIM_CUSTOMER d WHERE d.customer_id = s.customer_id AND d.is_current);

### 7-1b. SCD2 : DIM_PRODUCT

In [ ]:
-- 7-1. SCD2 : dim_product
UPDATE MART.DIM_PRODUCT
   SET valid_to = $run_ts, is_current = FALSE
  FROM CORE.PRODUCTS s
 WHERE MART.DIM_PRODUCT.product_id = s.product_id
   AND MART.DIM_PRODUCT.is_current
   AND MART.DIM_PRODUCT.product_key <> -1
   AND MART.DIM_PRODUCT.hash_diff <> s.hash_diff;

INSERT INTO MART.DIM_PRODUCT (product_id, product_name, category, valid_from, valid_to, is_current, hash_diff, load_id)
SELECT s.product_id, s.product_name, s.category,
       CASE WHEN h.product_id IS NULL THEN '1900-01-01'::TIMESTAMP_NTZ ELSE $run_ts END,
       NULL, TRUE, s.hash_diff, $load_id
FROM CORE.PRODUCTS s
LEFT JOIN (SELECT DISTINCT product_id FROM MART.DIM_PRODUCT) h ON h.product_id = s.product_id
WHERE NOT EXISTS (SELECT 1 FROM MART.DIM_PRODUCT d WHERE d.product_id = s.product_id AND d.is_current);

### 7-2a. FACT_SALES_LINE (MERGE)
주문일 시점에 유효했던 차원 버전으로 대리키를 연결(point-in-time). 없는 참조는 -1, 사라진 행은 `is_deleted`.

In [ ]:
-- 7-2. fact_sales_line
MERGE INTO MART.FACT_SALES_LINE AS t
USING (
  SELECT p.order_item_id, p.order_id,
         COALESCE(c.customer_key, -1)      AS customer_key,
         COALESCE(pr.product_key, -1)      AS product_key,
         COALESCE(d.date_key, -1)          AS order_date_key,
         COALESCE(s.order_status_key, -1)  AS order_status_key,
         p.order_date, p.quantity, p.unit_price, p.line_amount, p.is_revenue_eligible, p.exclusion_reason
  FROM CORE.SALES_LINE_PREP p
  LEFT JOIN MART.DIM_CUSTOMER c
         ON c.customer_id = p.customer_id AND c.customer_key <> -1
        AND CAST(p.order_date AS TIMESTAMP_NTZ) >= c.valid_from
        AND CAST(p.order_date AS TIMESTAMP_NTZ) <  COALESCE(c.valid_to, '9999-12-31'::TIMESTAMP_NTZ)
  LEFT JOIN MART.DIM_PRODUCT pr
         ON pr.product_id = p.product_id AND pr.product_key <> -1
        AND CAST(p.order_date AS TIMESTAMP_NTZ) >= pr.valid_from
        AND CAST(p.order_date AS TIMESTAMP_NTZ) <  COALESCE(pr.valid_to, '9999-12-31'::TIMESTAMP_NTZ)
  LEFT JOIN MART.DIM_DATE d          ON d.calendar_date = p.order_date
  LEFT JOIN MART.DIM_ORDER_STATUS s  ON s.status_code = p.order_status
) AS src
ON t.order_item_id = src.order_item_id
WHEN MATCHED THEN UPDATE SET
  order_id = src.order_id, customer_key = src.customer_key, product_key = src.product_key,
  order_date_key = src.order_date_key, order_status_key = src.order_status_key, order_date = src.order_date,
  quantity = src.quantity, unit_price = src.unit_price, line_amount = src.line_amount,
  is_revenue_eligible = src.is_revenue_eligible, exclusion_reason = src.exclusion_reason,
  is_deleted = FALSE, load_id = $load_id, updated_at = $run_ts
WHEN NOT MATCHED THEN INSERT
  (order_item_id, order_id, customer_key, product_key, order_date_key, order_status_key, order_date,
   quantity, unit_price, line_amount, is_revenue_eligible, exclusion_reason, is_deleted, load_id, updated_at)
VALUES
  (src.order_item_id, src.order_id, src.customer_key, src.product_key, src.order_date_key, src.order_status_key, src.order_date,
   src.quantity, src.unit_price, src.line_amount, src.is_revenue_eligible, src.exclusion_reason, FALSE, $load_id, $run_ts);

UPDATE MART.FACT_SALES_LINE SET is_deleted = TRUE, load_id = $load_id, updated_at = $run_ts
WHERE NOT is_deleted AND order_item_id NOT IN (SELECT order_item_id FROM CORE.SALES_LINE_PREP);

### 7-2b. FACT_PAYMENT (MERGE)

In [ ]:
-- 7-2. fact_payment
MERGE INTO MART.FACT_PAYMENT AS t
USING (
  SELECT p.payment_id, p.order_id,
         COALESCE(c.customer_key, -1)       AS customer_key,
         COALESCE(d.date_key, -1)           AS paid_date_key,
         COALESCE(s.payment_status_key, -1) AS payment_status_key,
         p.paid_at, p.payment_amount, p.has_valid_order
  FROM CORE.PAYMENT_PREP p
  LEFT JOIN MART.DIM_CUSTOMER c
         ON c.customer_id = p.customer_id AND c.customer_key <> -1
        AND CAST(p.order_date AS TIMESTAMP_NTZ) >= c.valid_from
        AND CAST(p.order_date AS TIMESTAMP_NTZ) <  COALESCE(c.valid_to, '9999-12-31'::TIMESTAMP_NTZ)
  LEFT JOIN MART.DIM_DATE d            ON d.calendar_date = p.paid_date
  LEFT JOIN MART.DIM_PAYMENT_STATUS s  ON s.status_code = p.payment_status
) AS src
ON t.payment_id = src.payment_id
WHEN MATCHED THEN UPDATE SET
  order_id = src.order_id, customer_key = src.customer_key, paid_date_key = src.paid_date_key,
  payment_status_key = src.payment_status_key, paid_at = src.paid_at, payment_amount = src.payment_amount,
  has_valid_order = src.has_valid_order, is_deleted = FALSE, load_id = $load_id, updated_at = $run_ts
WHEN NOT MATCHED THEN INSERT
  (payment_id, order_id, customer_key, paid_date_key, payment_status_key, paid_at, payment_amount, has_valid_order, is_deleted, load_id, updated_at)
VALUES
  (src.payment_id, src.order_id, src.customer_key, src.paid_date_key, src.payment_status_key, src.paid_at, src.payment_amount,
   src.has_valid_order, FALSE, $load_id, $run_ts);

UPDATE MART.FACT_PAYMENT SET is_deleted = TRUE, load_id = $load_id, updated_at = $run_ts
WHERE NOT is_deleted AND payment_id NOT IN (SELECT payment_id FROM CORE.PAYMENT_PREP);

### 7-2c. FACT_ORDER_LIFECYCLE (MERGE)

In [ ]:
-- 7-2. fact_order_lifecycle
MERGE INTO MART.FACT_ORDER_LIFECYCLE AS t
USING (
  SELECT q.order_id,
         COALESCE(c.customer_key, -1)      AS customer_key,
         COALESCE(d.date_key, -1)          AS order_date_key,
         COALESCE(pd.date_key, -1)         AS first_success_paid_date_key,
         COALESCE(s.order_status_key, -1)  AS order_status_key,
         q.order_date, q.first_success_paid_at, q.order_amount, q.line_count, q.line_amount_total,
         q.success_payment_count, q.success_payment_amount,
         DATEDIFF('day', q.order_date, CAST(q.first_success_paid_at AS DATE)) AS days_to_first_success_payment,
         q.is_revenue_eligible, q.exclusion_reason
  FROM CORE.ORDER_QUALITY q
  LEFT JOIN MART.DIM_CUSTOMER c
         ON c.customer_id = q.customer_id AND c.customer_key <> -1
        AND CAST(q.order_date AS TIMESTAMP_NTZ) >= c.valid_from
        AND CAST(q.order_date AS TIMESTAMP_NTZ) <  COALESCE(c.valid_to, '9999-12-31'::TIMESTAMP_NTZ)
  LEFT JOIN MART.DIM_DATE d          ON d.calendar_date  = q.order_date
  LEFT JOIN MART.DIM_DATE pd         ON pd.calendar_date = CAST(q.first_success_paid_at AS DATE)
  LEFT JOIN MART.DIM_ORDER_STATUS s  ON s.status_code    = q.order_status
) AS src
ON t.order_id = src.order_id
WHEN MATCHED THEN UPDATE SET
  customer_key = src.customer_key, order_date_key = src.order_date_key,
  first_success_paid_date_key = src.first_success_paid_date_key, order_status_key = src.order_status_key,
  order_date = src.order_date, first_success_paid_at = src.first_success_paid_at, order_amount = src.order_amount,
  line_count = src.line_count, line_amount_total = src.line_amount_total,
  success_payment_count = src.success_payment_count, success_payment_amount = src.success_payment_amount,
  days_to_first_success_payment = src.days_to_first_success_payment,
  is_revenue_eligible = src.is_revenue_eligible, exclusion_reason = src.exclusion_reason,
  is_deleted = FALSE, load_id = $load_id, updated_at = $run_ts
WHEN NOT MATCHED THEN INSERT
  (order_id, customer_key, order_date_key, first_success_paid_date_key, order_status_key, order_date, first_success_paid_at,
   order_amount, line_count, line_amount_total, success_payment_count, success_payment_amount, days_to_first_success_payment,
   is_revenue_eligible, exclusion_reason, is_deleted, load_id, updated_at)
VALUES
  (src.order_id, src.customer_key, src.order_date_key, src.first_success_paid_date_key, src.order_status_key, src.order_date,
   src.first_success_paid_at, src.order_amount, src.line_count, src.line_amount_total, src.success_payment_count,
   src.success_payment_amount, src.days_to_first_success_payment, src.is_revenue_eligible, src.exclusion_reason,
   FALSE, $load_id, $run_ts);

UPDATE MART.FACT_ORDER_LIFECYCLE SET is_deleted = TRUE, load_id = $load_id, updated_at = $run_ts
WHERE NOT is_deleted AND order_id NOT IN (SELECT order_id FROM CORE.ORDER_QUALITY);

### 7-3. 품질 검사 (17개)
`failures` 가 모두 0 이어야 합니다. 결과는 `AUDIT.DQ_RESULTS` 에 저장됩니다.

In [ ]:
-- 7-3. 품질 검사 (failures 가 모두 0 이어야 COMMIT)
DELETE FROM AUDIT.DQ_RESULTS WHERE load_id = $load_id;   -- 같은 배치를 다시 실행해도 결과가 중복되지 않게
INSERT INTO AUDIT.DQ_RESULTS (load_id, check_name, failures)
SELECT $load_id, check_name, failures FROM (
  SELECT 'dim_customer_one_current_version' AS check_name, COUNT(*) AS failures
    FROM (SELECT customer_id FROM MART.DIM_CUSTOMER GROUP BY customer_id HAVING SUM(CASE WHEN is_current THEN 1 ELSE 0 END) <> 1) x
  UNION ALL SELECT 'dim_customer_key_unique', COUNT(*)
    FROM (SELECT customer_key FROM MART.DIM_CUSTOMER GROUP BY customer_key HAVING COUNT(*) > 1) x
  UNION ALL SELECT 'dim_customer_valid_range', COUNT(*)
    FROM MART.DIM_CUSTOMER WHERE valid_to IS NOT NULL AND valid_to <= valid_from
  UNION ALL SELECT 'dim_product_one_current_version', COUNT(*)
    FROM (SELECT product_id FROM MART.DIM_PRODUCT GROUP BY product_id HAVING SUM(CASE WHEN is_current THEN 1 ELSE 0 END) <> 1) x
  UNION ALL SELECT 'dim_product_key_unique', COUNT(*)
    FROM (SELECT product_key FROM MART.DIM_PRODUCT GROUP BY product_key HAVING COUNT(*) > 1) x
  UNION ALL SELECT 'fact_sales_line_key_unique', COUNT(*)
    FROM (SELECT order_item_id FROM MART.FACT_SALES_LINE GROUP BY order_item_id HAVING COUNT(*) > 1) x
  UNION ALL SELECT 'fact_sales_line_customer_fk', COUNT(*)
    FROM MART.FACT_SALES_LINE f LEFT JOIN MART.DIM_CUSTOMER c ON f.customer_key = c.customer_key WHERE c.customer_key IS NULL
  UNION ALL SELECT 'fact_sales_line_product_fk', COUNT(*)
    FROM MART.FACT_SALES_LINE f LEFT JOIN MART.DIM_PRODUCT p ON f.product_key = p.product_key WHERE p.product_key IS NULL
  UNION ALL SELECT 'fact_sales_line_date_fk', COUNT(*)
    FROM MART.FACT_SALES_LINE f LEFT JOIN MART.DIM_DATE d ON f.order_date_key = d.date_key WHERE d.date_key IS NULL
  UNION ALL SELECT 'fact_sales_line_status_fk', COUNT(*)
    FROM MART.FACT_SALES_LINE f LEFT JOIN MART.DIM_ORDER_STATUS s ON f.order_status_key = s.order_status_key WHERE s.order_status_key IS NULL
  UNION ALL SELECT 'fact_sales_line_amount_calc', COUNT(*)
    FROM MART.FACT_SALES_LINE WHERE quantity <= 0 OR unit_price < 0 OR line_amount <> quantity * unit_price
  UNION ALL SELECT 'fact_payment_key_unique', COUNT(*)
    FROM (SELECT payment_id FROM MART.FACT_PAYMENT GROUP BY payment_id HAVING COUNT(*) > 1) x
  UNION ALL SELECT 'fact_payment_status_fk', COUNT(*)
    FROM MART.FACT_PAYMENT f LEFT JOIN MART.DIM_PAYMENT_STATUS s ON f.payment_status_key = s.payment_status_key WHERE s.payment_status_key IS NULL
  UNION ALL SELECT 'fact_order_lifecycle_key_unique', COUNT(*)
    FROM (SELECT order_id FROM MART.FACT_ORDER_LIFECYCLE GROUP BY order_id HAVING COUNT(*) > 1) x
  UNION ALL SELECT 'revenue_order_reconciliation', COUNT(*)
    FROM MART.FACT_ORDER_LIFECYCLE
   WHERE is_revenue_eligible AND NOT is_deleted
     AND (order_amount <> line_amount_total OR order_amount <> success_payment_amount OR success_payment_count <> 1)
  UNION ALL SELECT 'revenue_line_vs_order', COUNT(*)
    FROM MART.FACT_ORDER_LIFECYCLE o
    LEFT JOIN (SELECT order_id, SUM(line_amount) AS amt FROM MART.FACT_SALES_LINE WHERE NOT is_deleted GROUP BY order_id) l
           ON o.order_id = l.order_id
   WHERE o.is_revenue_eligible AND NOT o.is_deleted AND (l.amt IS NULL OR l.amt <> o.order_amount)
  UNION ALL SELECT 'revenue_requires_known_keys', COUNT(*)
    FROM MART.FACT_SALES_LINE
   WHERE is_revenue_eligible AND NOT is_deleted AND (customer_key = -1 OR product_key = -1 OR order_date_key = -1)
) checks;

In [ ]:
SELECT check_name, failures FROM AUDIT.DQ_RESULTS WHERE load_id = $load_id ORDER BY failures DESC, check_name;   -- 전부 0

### 7-4. 게이트 : 배치 상태 확정
검사 실패가 하나라도 있으면 `FAILED`, 모두 통과하면 `SUCCESS` 로 `AUDIT.PIPELINE_RUNS` 를 갱신합니다.

In [ ]:
UPDATE AUDIT.PIPELINE_RUNS p
   SET status = IFF(q.fail_cnt = 0, 'SUCCESS', 'FAILED'),
       finished_at = CURRENT_TIMESTAMP()::TIMESTAMP_NTZ,
       message = IFF(q.fail_cnt = 0, NULL, 'DQ failed: ' || q.fail_names),
       input_rows = s.input_rows, duplicate_rows = s.duplicate_rows, quarantined_rows = s.quarantined_rows,
       flagged_issues = s.flagged_issues, revenue_orders = s.revenue_orders, revenue_amount = s.revenue_amount
  FROM CORE.RUN_SUMMARY s,
       (SELECT COALESCE(SUM(failures), 0) AS fail_cnt,
               COALESCE(LISTAGG(CASE WHEN failures > 0 THEN check_name END, ', '), '') AS fail_names
          FROM AUDIT.DQ_RESULTS WHERE load_id = $load_id) q
 WHERE p.load_id = $load_id;

SELECT load_id, status, message, input_rows, revenue_orders, revenue_amount
FROM AUDIT.PIPELINE_RUNS WHERE load_id = $load_id;   -- 기대값: SUCCESS / 27,055 / 2,198,198,700

## D. 결과 확인

In [ ]:
SELECT 'DIM_CUSTOMER' AS tbl, COUNT(*) AS n FROM MART.DIM_CUSTOMER WHERE customer_key <> -1   -- 기대값 1,998
UNION ALL SELECT 'DIM_PRODUCT', COUNT(*) FROM MART.DIM_PRODUCT WHERE product_key <> -1             -- 151
UNION ALL SELECT 'DIM_DATE', COUNT(*) FROM MART.DIM_DATE                                           -- 4,019
UNION ALL SELECT 'FACT_SALES_LINE', COUNT(*) FROM MART.FACT_SALES_LINE                             -- 65,789
UNION ALL SELECT 'FACT_PAYMENT', COUNT(*) FROM MART.FACT_PAYMENT                                   -- 30,966
UNION ALL SELECT 'FACT_ORDER_LIFECYCLE', COUNT(*) FROM MART.FACT_ORDER_LIFECYCLE;                  -- 29,802

SELECT COUNT(*) AS revenue_orders, SUM(order_amount) AS revenue
FROM MART.FACT_ORDER_LIFECYCLE WHERE is_revenue_eligible AND NOT is_deleted;   -- 27,055 / 2,198,198,700